# Consensus Principle Selection

In [1]:
from pathlib import Path
from typing import Dict, List, Tuple

import time
import sys
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm
from sklearn.cluster import AgglomerativeClustering


In [2]:
DEDUPLICATED_DATA_PATH = Path(r"C:\Users\laras\OneDrive\Dokumente\duke_classes\icai_project\Principle-Elicitation\deduplication\intermediate_principle_clusters_k=1371.json")

TOP_K_PRINCIPLES = 5
K_SUMMARY = 5

TRESHOLD = "k=1371"

SUMMARIZED_DATA_PATH = f"cluster_summarized_{TRESHOLD}_fair_clustering_deduplicated.json"


# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=api_key)

In [3]:
def summarize_cluster(cluster, prompt, summary_prompt):
    num_retries = 0
    max_retries = 5
    while True:
        try:
            response = client.chat.completions.create(
                model="gpt-4.1-2025-04-14",
                messages=[
                    {"role": "system", "content": summary_prompt},
                    {"role": "user", "content": prompt}
                ],
            )
            summary = response.choices[0].message.content.strip()
            return({
                "cluster_id": cluster["cluster_id"],
                "summarized_principle": summary,
                "original_principles": cluster["principles"]
            })
            break
        except Exception as e:
            print(f"Error: {e}. Retrying in 5 seconds...")
            num_retries += 1
            if num_retries >= max_retries:
                print("Max retries reached. Exiting.")
                sys.exit(1)
            time.sleep(5)

In [4]:
def embed_texts(texts):
    single_input = isinstance(texts, str)
    if single_input:
        texts = [texts]

    batch_size = 256  
    all_vectors = []
    max_retries = 5

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        num_retries = 0
        while True:
            try:
                resp = client.embeddings.create(
                    model="text-embedding-3-small",
                    input=batch,
                )
                all_vectors.extend([item.embedding for item in resp.data])
                break
            except Exception as e:
                num_retries += 1
                print(f"Error: {e}. Retrying in 5 seconds...")
                if num_retries >= max_retries:
                    print("Max retries reached. Exiting.")
                    sys.exit(1)
                time.sleep(5)

    arr = np.array(all_vectors, dtype=float)

    return arr

In [5]:
def extract_principle_strings(cluster_item):
    out = []
    for p in cluster_item.get("principles", []):
        if isinstance(p, (list, tuple)) and len(p) > 0:
            s = (p[0] or "").strip()
        else:
            s = (p or "").strip()
        if s:
            out.append(s)
    return out

In [6]:
with open('principle_summary_prompt.txt', 'r', encoding="utf-8") as f:
    summary_prompt = f.read()

In [7]:
with DEDUPLICATED_DATA_PATH.open("r", encoding="utf-8") as f:
    clusters_output = json.load(f)

all_principles = []
cluster_offsets = []
for cluster in clusters_output:
    principles = [p[0].strip() for p in cluster.get("principles", []) if isinstance(p, (list, tuple)) and p and p[0] and str(p[0]).strip()]
    cluster_offsets.append((len(all_principles), len(all_principles) + len(principles)))
    all_principles.extend(principles)

all_embeddings = embed_texts(all_principles)

summarized_principles = []

for cluster, (a, b) in tqdm(list(zip(clusters_output, cluster_offsets)), total=len(clusters_output)):
    principles = all_principles[a:b]
    if not principles:
        continue

    cluster_embeddings = all_embeddings[a:b]
    centroid = np.mean(cluster_embeddings, axis=0)

    distances = np.linalg.norm(cluster_embeddings - centroid, axis=1)
    closest_indices = np.argsort(distances)[:K_SUMMARY]
    closest_principles = [principles[i] for i in closest_indices]

    principles_text = "\n".join(f"{i+1}. {p}" for i, p in enumerate(closest_principles))
    prompt = f"\n\nPrinciples:\n{principles_text}\n\nSummarized Principle:"

    summarized_principles.append(summarize_cluster(cluster, prompt, summary_prompt))

with open(SUMMARIZED_DATA_PATH, "w", encoding="utf-8") as f:
    json.dump(summarized_principles, f, indent=4, ensure_ascii=False)


100%|██████████████████████████████████████████████████████████████████████████████████| 49/49 [00:34<00:00,  1.40it/s]


In [8]:
summaries = []
summary_cluster_ids = []
originals = []

for idx, item in enumerate(summarized_principles):
    cluster_id = item.get("cluster_id")

    summarized_principle = item.get("summarized_principle").strip()
    principle_list = item.get("original_principles", [])

    if summarized_principle:
        summaries.append(summarized_principle)
        summary_cluster_ids.append(cluster_id)

    for p in principle_list:
        p = (p[0] if isinstance(p, (list, tuple)) and p else p)
        p = (p or "").strip()
        if p:
            originals.append(p)

print(f"#summaries: {len(summaries)}")
print(f"#originals: {len(originals)}")


#summaries: 49
#originals: 1129


In [9]:
emb_orig = embed_texts(originals)  
emb_summaries = embed_texts(summaries)  

N, d = emb_orig.shape if emb_orig.size else (0, 0)
M, d_s = emb_summaries.shape if emb_summaries.size else (0, 0)

print(f"emb_orig shape: {emb_orig.shape}")
print(f"emb_summaries shape: {emb_summaries.shape}")

emb_orig shape: (1129, 1536)
emb_summaries shape: (49, 1536)


In [10]:
global_msd = []  

for j in range(M):
    s_j = emb_summaries[j] 
    diff = emb_orig - s_j   
    msd_j = (diff ** 2).mean() 
    global_msd.append(float(msd_j))

global_msd = np.array(global_msd)
global_msd[:10]  

array([0.00046995, 0.00045229, 0.00046865, 0.00045012, 0.00047531,
       0.00052572, 0.0004354 , 0.00047278, 0.0004798 , 0.00050446])

In [11]:
idx_sorted = np.argsort(global_msd)

rows = []
for rank, j in enumerate(idx_sorted, start=1):
    rows.append(
        {
            "rank": rank,
            "cluster_id": summary_cluster_ids[j],
            "summary_text": summaries[j],
            "global_msd": float(global_msd[j]),
        }
    )

df_summaries = pd.DataFrame(rows)
df_summaries.head(TOP_K_PRINCIPLES)

,rank,cluster_id,summary_text,global_msd
0,1,41,AI should consistently offer factual and accur...,0.000411
1,2,13,"AI should interact in a friendly, kind, and re...",0.000432
2,3,6,"AI should provide clear, precise, and accurate...",0.000435
3,4,3,AI should avoid providing false or harmful inf...,0.000450
4,5,1,"AI should act impartially and without bias, up...",0.000452


In [12]:
output_json_path = Path(f"consensus_principle_selection_{TRESHOLD}_fair_clustering.json")
with output_json_path.open("w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)

print("Saved JSON to:", output_json_path)

Saved JSON to: consensus_principle_selection_k=1371_fair_clustering.json


In [13]:
top = df_summaries.head(10)
for _, row in top.iterrows():
    print("Rank:", row["rank"])
    print("Cluster:", row["cluster_id"])
    print("Global MSD:", row["global_msd"])
    print("Summary:\n", row["summary_text"])
    print("-" * 80)

Rank: 1
Cluster: 41
Global MSD: 0.0004111590891299402
Summary:
 AI should consistently offer factual and accurate information.
--------------------------------------------------------------------------------
Rank: 2
Cluster: 13
Global MSD: 0.0004321645699897715
Summary:
 AI should interact in a friendly, kind, and respectful manner.
--------------------------------------------------------------------------------
Rank: 3
Cluster: 6
Global MSD: 0.0004354001416805469
Summary:
 AI should provide clear, precise, and accurate information in its responses.
--------------------------------------------------------------------------------
Rank: 4
Cluster: 3
Global MSD: 0.0004501154687772052
Summary:
 AI should avoid providing false or harmful information to anyone.
--------------------------------------------------------------------------------
Rank: 5
Cluster: 1
Global MSD: 0.00045228576656215004
Summary:
 AI should act impartially and without bias, upholding ethical standards in its behavior.


In [14]:
def select_top_k_with_similarity_aggregation(
    ranked_df,
    embed_texts,
    top_k=10,
    distance_threshold=0.3,
    text_col="summary_text",
    score_col="global_msd",
    cluster_id_col="cluster_id",
):
    df = ranked_df.sort_values(by=score_col, ascending=True).reset_index(drop=True)

    texts = [str(t).strip() for t in df[text_col].tolist()]
    emb = np.asarray(embed_texts(texts), dtype=float)
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    emb = emb / norms

    clustering_model = AgglomerativeClustering(
        n_clusters=None,
        linkage="complete",
        metric="cosine",
        distance_threshold=distance_threshold,
    )
    labels = clustering_model.fit_predict(emb)

    groups = {}
    for i, lab in enumerate(labels):
        groups.setdefault(int(lab), []).append(i)

    ordered_labels = []
    seen = set()
    for i in range(len(df)):
        lab = int(labels[i])
        if lab not in seen:
            ordered_labels.append(lab)
            seen.add(lab)
        if len(ordered_labels) >= top_k:
            break

    selected = []
    for rank, lab in enumerate(ordered_labels, start=1):
        idxs = groups[lab]
        rep_idx = min(idxs)
        rep_row = df.iloc[rep_idx]

        aggregated_items = []
        aggregated_cluster_ids = []
        for j in idxs:
            r = df.iloc[j]
            aggregated_cluster_ids.append(int(r[cluster_id_col]))
            aggregated_items.append({
                "cluster_id": r[cluster_id_col],
                "global_msd": float(r[score_col]),
                "summary_text": texts[j],
            })

        selected.append({
            "rank": rank,
            "cluster_id": rep_row[cluster_id_col],
            "global_msd": float(rep_row[score_col]),
            "summary_text": texts[rep_idx],
            "aggregated_cluster_ids": aggregated_cluster_ids,
            "aggregated_count": len(idxs),
            "aggregated_items": aggregated_items,
        })

    return selected



In [16]:
selected_top10 = select_top_k_with_similarity_aggregation(
    ranked_df=df_summaries,
    embed_texts=embed_texts,
    top_k=10,
    distance_threshold=0.3,
    text_col="summary_text",
    score_col="global_msd",
    cluster_id_col="cluster_id",
)

for item in selected_top10:
    print("Rank:", item["rank"])
    print("Cluster:", item["cluster_id"])
    print("Global MSD:", item["global_msd"])
    print("Aggregated Count:", item["aggregated_count"])
    print("Aggregated Cluster IDs:", item["aggregated_cluster_ids"])
    print("Summary:\n", item["summary_text"])
    print("-" * 80)


Rank: 1
Cluster: 41
Global MSD: 0.0004111590891299402
Aggregated Count: 5
Aggregated Cluster IDs: [41, 6, 7, 38, 31]
Summary:
 AI should consistently offer factual and accurate information.
--------------------------------------------------------------------------------
Rank: 2
Cluster: 13
Global MSD: 0.0004321645699897715
Aggregated Count: 2
Aggregated Cluster IDs: [13, 4]
Summary:
 AI should interact in a friendly, kind, and respectful manner.
--------------------------------------------------------------------------------
Rank: 3
Cluster: 3
Global MSD: 0.0004501154687772052
Aggregated Count: 3
Aggregated Cluster IDs: [3, 2, 21]
Summary:
 AI should avoid providing false or harmful information to anyone.
--------------------------------------------------------------------------------
Rank: 4
Cluster: 1
Global MSD: 0.00045228576656215004
Aggregated Count: 2
Aggregated Cluster IDs: [1, 16]
Summary:
 AI should act impartially and without bias, upholding ethical standards in its behavior.

In [17]:
OUTPUT_PATH = Path(f"consensus_principle_selection_{TRESHOLD}_fair_clustering_aggregated.json"
)

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(selected_top10, f, indent=4, ensure_ascii=False, default=lambda o: int(o) if isinstance(o, np.integer) else float(o))

print("Saved JSON to:", OUTPUT_PATH)



Saved JSON to: consensus_principle_selection_k=1371_fair_clustering_aggregated.json
